In [1]:
import pandas as pd
import os

BASE_PATH = ".."
RAW_PATH = os.path.join(BASE_PATH, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")

TRANS_PATH = os.path.join(RAW_PATH, "LI-Small_Trans.csv")
ACCOUNTS_PATH = os.path.join(RAW_PATH, "LI-Small_accounts.csv")

trans_df = pd.read_csv(TRANS_PATH)
accounts_df = pd.read_csv(ACCOUNTS_PATH)

print("Transactions:", trans_df.shape)
print("Accounts:", accounts_df.shape)

Transactions: (6924049, 11)
Accounts: (712688, 5)


In [2]:
# Sender-side information
sender_currency = (
    trans_df.groupby("From Bank")["Receiving Currency"]
    .agg(
        Sender_Transaction_Count="count",
        Sender_Receiving_Currencies=lambda x: ", ".join(sorted(x.dropna().unique())),
        Most_Frequent_Receiving_Currency=lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"
    )
    .reset_index()
    .rename(columns={"From Bank": "Bank ID"})
)

# Receiver-side information
receiver_currency = (
    trans_df.groupby("To Bank")["Payment Currency"]
    .agg(
        Receiver_Transaction_Count="count",
        Receiver_Payment_Currencies=lambda x: ", ".join(sorted(x.dropna().unique())),
        Most_Frequent_Payment_Currency=lambda x: x.mode().iloc[0] if not x.mode().empty else "Unknown"
    )
    .reset_index()
    .rename(columns={"To Bank": "Bank ID"})
)

# Get bank names from accounts dataset
bank_names = (
    accounts_df[["Bank ID", "Bank Name"]]
    .drop_duplicates("Bank ID")
)

banks_df = bank_names.merge(
    sender_currency,
    on="Bank ID",
    how="outer"
)

banks_df = banks_df.merge(
    receiver_currency,
    on="Bank ID",
    how="outer"
)

# Fill banks that only appear on one side
banks_df["Sender_Transaction_Count"] = (
    banks_df["Sender_Transaction_Count"].fillna(0).astype("int64")
)

banks_df["Receiver_Transaction_Count"] = (
    banks_df["Receiver_Transaction_Count"].fillna(0).astype("int64")
)

banks_df["Sender_Receiving_Currencies"] = (
    banks_df["Sender_Receiving_Currencies"].fillna("")
)

banks_df["Receiver_Payment_Currencies"] = (
    banks_df["Receiver_Payment_Currencies"].fillna("")
)

banks_df["Most_Frequent_Receiving_Currency"] = (
    banks_df["Most_Frequent_Receiving_Currency"].fillna("Unknown")
)

banks_df["Most_Frequent_Payment_Currency"] = (
    banks_df["Most_Frequent_Payment_Currency"].fillna("Unknown")
)

print(banks_df.head())
print("Banks created:", len(banks_df))

   Bank ID               Bank Name  Sender_Transaction_Count  \
0        0          Italy Bank #18                     31992   
1        1  First Bank of Portland                     33245   
2        2          India Bank #46                     14325   
3        3       Germany Bank #106                     32061   
4        4          Japan Bank #40                     11880   

                         Sender_Receiving_Currencies  \
0  Australian Dollar, Bitcoin, Brazil Real, Euro,...   
1  Australian Dollar, Bitcoin, Brazil Real, Euro,...   
2  Bitcoin, Euro, Ruble, Rupee, UK Pound, US Doll...   
3  Australian Dollar, Bitcoin, Canadian Dollar, E...   
4              Canadian Dollar, Euro, US Dollar, Yen   

  Most_Frequent_Receiving_Currency  Receiver_Transaction_Count  \
0                             Euro                       15698   
1                        US Dollar                       15778   
2                            Rupee                        8506   
3             

In [3]:
# Combine currencies observed on sender and receiver sides
def get_top_combined_currencies(row):
    currencies = []

    if row["Sender_Receiving_Currencies"]:
        currencies.extend(row["Sender_Receiving_Currencies"].split(", "))

    if row["Receiver_Payment_Currencies"]:
        currencies.extend(row["Receiver_Payment_Currencies"].split(", "))

    counts = pd.Series(currencies).value_counts()

    return ", ".join(counts.head(3).index)


banks_df["Top_3_Combined_Recurring_Currencies"] = banks_df.apply(
    get_top_combined_currencies,
    axis=1
)

# Currency -> country/region proxy
currency_country = {
    "US Dollar": "United States",
    "Euro": "Eurozone",
    "Yuan": "China",
    "Yen": "Japan",
    "Rupee": "India",
    "Saudi Riyal": "Saudi Arabia",
    "UK Pound": "United Kingdom",
    "Australian Dollar": "Australia",
    "Canadian Dollar": "Canada",
    "Swiss Franc": "Switzerland",
    "Hong Kong Dollar": "Hong Kong",
    "Singapore Dollar": "Singapore",
    "UAE Dirham": "United Arab Emirates",
    "Bitcoin": "No Single Country"
}

banks_df["Country_of_Top_Currency"] = (
    banks_df["Most_Frequent_Receiving_Currency"]
    .map(currency_country)
    .fillna("Unknown")
)

# Rename columns to final project names
banks_df = banks_df.rename(columns={
    "Sender_Transaction_Count": "Sender Transaction Count",
    "Sender_Receiving_Currencies": "Sender Receiving Currencies",
    "Most_Frequent_Receiving_Currency": "Most Frequent Receiving Currency",
    "Receiver_Transaction_Count": "Receiver Transaction Count",
    "Receiver_Payment_Currencies": "Receiver Payment Currencies",
    "Most_Frequent_Payment_Currency": "Most Frequent Payment Currency",
    "Top_3_Combined_Recurring_Currencies": "Top 3 Combined Recurring Currencies",
    "Country_of_Top_Currency": "Country of Top Currency"
})

# Final column order
banks_df = banks_df[
    [
        "Bank ID",
        "Bank Name",
        "Sender Transaction Count",
        "Sender Receiving Currencies",
        "Most Frequent Receiving Currency",
        "Receiver Transaction Count",
        "Receiver Payment Currencies",
        "Most Frequent Payment Currency",
        "Top 3 Combined Recurring Currencies",
        "Country of Top Currency"
    ]
]

print(banks_df.head(10))

   Bank ID                Bank Name  Sender Transaction Count  \
0        0           Italy Bank #18                     31992   
1        1   First Bank of Portland                     33245   
2        2           India Bank #46                     14325   
3        3        Germany Bank #106                     32061   
4        4           Japan Bank #40                     11880   
5        5           India Bank #26                     13945   
6        7              UK Bank #19                      9473   
7        8           Canada Bank #0                     19691   
8       10          Canada Bank #18                     24421   
9       11  Savings Bank of Madison                    123456   

                         Sender Receiving Currencies  \
0  Australian Dollar, Bitcoin, Brazil Real, Euro,...   
1  Australian Dollar, Bitcoin, Brazil Real, Euro,...   
2  Bitcoin, Euro, Ruble, Rupee, UK Pound, US Doll...   
3  Australian Dollar, Bitcoin, Canadian Dollar, E...   
4   

In [4]:
expected_columns = [
    "Bank ID",
    "Bank Name",
    "Sender Transaction Count",
    "Sender Receiving Currencies",
    "Most Frequent Receiving Currency",
    "Receiver Transaction Count",
    "Receiver Payment Currencies",
    "Most Frequent Payment Currency",
    "Top 3 Combined Recurring Currencies",
    "Country of Top Currency"
]

print("Shape:", banks_df.shape)
print("\nColumns:")
print(banks_df.columns.tolist())

print("\nMissing values:")
print(banks_df.isnull().sum())

print("\nUnique Bank IDs:", banks_df["Bank ID"].nunique())
print("Duplicate Bank IDs:", banks_df["Bank ID"].duplicated().sum())

assert banks_df.columns.tolist() == expected_columns
assert banks_df["Bank ID"].duplicated().sum() == 0

OUTPUT_PATH = os.path.join(PROCESSED_PATH, "banks.csv")
banks_df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)

Shape: (41815, 10)

Columns:
['Bank ID', 'Bank Name', 'Sender Transaction Count', 'Sender Receiving Currencies', 'Most Frequent Receiving Currency', 'Receiver Transaction Count', 'Receiver Payment Currencies', 'Most Frequent Payment Currency', 'Top 3 Combined Recurring Currencies', 'Country of Top Currency']

Missing values:
Bank ID                                0
Bank Name                              0
Sender Transaction Count               0
Sender Receiving Currencies            0
Most Frequent Receiving Currency       0
Receiver Transaction Count             0
Receiver Payment Currencies            0
Most Frequent Payment Currency         0
Top 3 Combined Recurring Currencies    0
Country of Top Currency                0
dtype: int64

Unique Bank IDs: 41815
Duplicate Bank IDs: 0

Saved: ..\data\processed\banks.csv
